# Séance 2 — Régression Linéaire & Descente de Gradient (CORRIGÉ)

**Formation ML — formation-ml**

Version complète, publiée après la séance. Pour la version à trous, voir `notebook_pratique_regression.ipynb`.

### Objectifs de la séance

- Comprendre pourquoi on a besoin d'une **fonction de perte** pour mesurer une erreur de prédiction
- Passer de l'intuition « tracer la meilleure droite » à sa formulation mathématique (MSE)
- Découvrir deux façons de trouver cette droite : la **solution analytique** (équations normales) et la **descente de gradient**
- Implémenter la descente de gradient **à la main**, en NumPy, puis la comparer à scikit-learn
- Comprendre le rôle du **learning rate** dans la convergence

---
<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
On reprend où on s'était arrêté en Séance 1 : le revenu médian (<code>MedInc</code>) semblait lié au prix du logement (<code>MedHouseVal</code>). On s'était demandé comment tracer, de façon systématique, la droite qui représente le mieux cette relation. C'est le sujet du jour.
</div>

## Le cadre général : ①②③

Tout algorithme d'apprentissage supervisé se construit avec 3 briques, toujours les mêmes :

| Brique | Rôle |
|---|---|
| ① **Modèle** | une fonction qui produit une prédiction $\hat{y}$ |
| ② **Fonction de coût** | mesure à quel point $\hat{y}$ est loin de $y$ |
| ③ **Optimisation** | ajuste les paramètres pour minimiser le coût |

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
Ce découpage revient à chaque séance du module — seul le contenu de chaque brique change. Aujourd'hui : ① une droite, ② la MSE, ③ équations normales ou descente de gradient.
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D

sns.set_theme(style="whitegrid")
np.random.seed(42)

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

# Pour cette séance, on travaille d'abord sur UNE seule feature pour que tout
# reste visualisable en 2D. On généralisera au multi-features à la fin.
X_simple = df[["MedInc"]].values
y = df["MedHouseVal"].values

X_simple.shape, y.shape

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_simple, y, alpha=0.15, s=15)
plt.xlabel("Revenu médian (10k $)")
plt.ylabel("Valeur médiane du logement (100k $)")
plt.title("Ce qu'on voyait déjà en Séance 1")
plt.show()

## 1. Le modèle linéaire

On cherche une fonction qui prédit $\hat{y}$ à partir de $x$ (le revenu médian) :

$$\hat{y} = w \cdot x + b$$

- $w$ (le **poids**) : la pente de la droite
- $b$ (le **biais**) : l'ordonnée à l'origine
- $\hat{y}$ : la prédiction du modèle (à distinguer de $y$, la vraie valeur observée)

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
$w$ et $b$ sont les <b>paramètres</b> du modèle — ce sont exactement eux que l'entraînement va chercher à ajuster. Avant l'entraînement, on ne sait pas quelles valeurs leur donner : c'est tout le problème.
</div>

In [ ]:
# Exemple avec des paramètres choisis au hasard, pour voir à quoi ça ressemble
w_test, b_test = 0.3, 0.5

x_line = np.linspace(X_simple.min(), X_simple.max(), 100)
y_line = w_test * x_line + b_test

plt.figure(figsize=(7, 5))
plt.scatter(X_simple, y, alpha=0.15, s=15, label="données")
plt.plot(x_line, y_line, color="red", linewidth=2, label=f"w={w_test}, b={b_test} (au hasard)")
plt.xlabel("Revenu médian (10k $)")
plt.ylabel("Valeur médiane du logement (100k $)")
plt.legend()
plt.title("Une droite choisie au hasard — clairement pas optimale")
plt.show()

<div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Attention</b><br>
Cette droite n'est pas complètement absurde, mais elle est clairement sous-optimale. Le but de la suite : trouver <i>automatiquement</i> le <b>meilleur</b> $w$ et le meilleur $b$, plutôt que de les deviner.
</div>

## 2. Comment mesurer si une droite est "bonne" ?

Il faut une façon **chiffrée** de dire "cette droite est meilleure que celle-là". C'est le rôle de la **fonction de perte** (*loss function*).

### L'erreur quadratique moyenne (MSE)

Pour chaque point $i$, l'erreur est la différence entre la vraie valeur $y_i$ et la prédiction $\hat{y}_i$ :

$$e_i = y_i - \hat{y}_i$$

On ne peut pas juste sommer les erreurs (elles se compenseraient : une erreur de +5 et une de -5 donneraient 0, comme si tout allait bien). On les élève au carré, puis on fait la moyenne sur les $n$ exemples :

$$\text{MSE}(w, b) = \frac{1}{n} \sum_{i=1}^{n} \left( y_i - (w x_i + b) \right)^2$$

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
La MSE est une fonction de $w$ et $b$ — <b>pas</b> de $x$. Pour chaque couple $(w, b)$ possible, elle donne un seul chiffre : à quel point cette droite se trompe en moyenne, au carré. On veut trouver le $(w, b)$ qui <b>minimise</b> ce chiffre.
</div>

In [ ]:
def mse(w, b, X, y):
    y_pred = w * X.flatten() + b
    return np.mean((y - y_pred) ** 2)

# Comparons deux droites
print("MSE avec w=0.3, b=0.5  :", mse(0.3, 0.5, X_simple, y))
print("MSE avec w=0.4, b=0.4  :", mse(0.4, 0.4, X_simple, y))

<div style="background:#f3e5f5;border-left:5px solid #8e24aa;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✏️ Exercice</b><br>
Modifie les valeurs de <code>w</code> et <code>b</code> ci-dessus et relance la cellule plusieurs fois. Essaie de trouver, à tâtons, une combinaison qui donne une MSE plus basse que les deux proposées. <b>Note ta meilleure MSE trouvée</b> — on va comparer avec la solution automatique plus loin.
</div>

In [ ]:
w_essai = 0.42
b_essai = 0.45

mse(w_essai, b_essai, X_simple, y)

## 3. Visualiser la fonction de perte

La MSE dépend de deux paramètres ($w$ et $b$) : on peut donc la représenter comme une **surface** en 3D, ou comme des **courbes de niveau** en 2D. C'est cette surface qu'on va "descendre".

In [ ]:
w_range = np.linspace(-0.5, 1.5, 80)
b_range = np.linspace(-2, 3, 80)
W, B = np.meshgrid(w_range, b_range)

Z = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        Z[i, j] = mse(W[i, j], B[i, j], X_simple, y)

fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(W, B, Z, cmap="viridis", alpha=0.85)
ax1.set_xlabel("w")
ax1.set_ylabel("b")
ax1.set_zlabel("MSE")
ax1.set_title("La surface de perte (3D)")

ax2 = fig.add_subplot(1, 2, 2)
contour = ax2.contour(W, B, Z, levels=30, cmap="viridis")
ax2.set_xlabel("w")
ax2.set_ylabel("b")
ax2.set_title("Les mêmes courbes de niveau (2D)")
plt.colorbar(contour, ax=ax2)

plt.tight_layout()
plt.show()

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
Le point le plus bas de cette surface (le centre des courbes de niveau concentriques) correspond au couple $(w, b)$ qui minimise la MSE — <b>c'est exactement ce qu'on cherche</b>. La forme en "bol" n'est pas un hasard : la MSE d'un modèle linéaire est une fonction <b>convexe</b>, donc elle n'a qu'un seul minimum, pas de fausses vallées où on pourrait rester coincé.
</div>

## 3bis. La MSE n'est pas la seule fonction de coût possible

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
La MSE est le choix standard en régression, mais pas le seul. La <b>MAE</b> (erreur absolue moyenne) est une alternative : $\text{MAE} = \frac{1}{n}\sum |y_i - \hat{y}_i|$. Comparons leur comportement.
</div>

In [ ]:
def mae(w, b, X, y):
    y_pred = w * X.flatten() + b
    return np.mean(np.abs(y - y_pred))

# Comparaison visuelle : comment chaque fonction réagit à une erreur
e = np.linspace(-4, 4, 200)
plt.figure(figsize=(7, 5))
plt.plot(e, e**2, label="MSE : $e^2$", linewidth=2.5)
plt.plot(e, np.abs(e), label="MAE : $|e|$", linewidth=2.5)
plt.xlabel("Erreur $e = y - \hat{y}$")
plt.ylabel("Coût")
plt.title("MSE pénalise les grosses erreurs bien plus fort que la MAE")
plt.legend()
plt.show()

In [ ]:
# Sensibilité aux outliers : on ajoute un point aberrant à fort effet de levier
from scipy.optimize import minimize

x_sample = X_simple[:150].flatten()
y_sample = y[:150]
x_out = np.append(x_sample, 14.5)
y_out = np.append(y_sample, 0.4)  # valeur très basse pour un revenu très élevé -> aberrant

def mse_loss(params, x, y):
    w, b = params
    return np.mean((y - (w * x + b)) ** 2)

def mae_loss(params, x, y):
    w, b = params
    return np.mean(np.abs(y - (w * x + b)))

res_mse = minimize(mse_loss, x0=[0.4, 0.5], args=(x_out, y_out))
res_mae = minimize(mae_loss, x0=[0.4, 0.5], args=(x_out, y_out), method="Nelder-Mead")

print("Pente trouvée avec MSE :", round(res_mse.x[0], 3))
print("Pente trouvée avec MAE :", round(res_mae.x[0], 3))

<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
La pente trouvée avec la MSE est davantage tirée vers l'outlier que celle trouvée avec la MAE — la MSE est plus <b>sensible aux valeurs aberrantes</b> car elle pénalise les grosses erreurs au carré. C'est un vrai choix de modélisation, pas un détail technique.
</div>

## 4. Une première solution : les équations normales

Pour la régression linéaire précisément, il existe une formule **directe** qui donne le $(w, b)$ optimal, sans itération. En notation matricielle (avec $X$ incluant une colonne de 1 pour le biais) :

$$\boldsymbol{\theta}^* = (X^T X)^{-1} X^T y$$

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
C'est la <b>solution des moindres carrés</b> — elle tombe directement au fond du bol qu'on vient de visualiser, en une seule opération matricielle. Pas besoin de deviner ni d'itérer.
</div><div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Attention</b><br>
Cette formule demande d'inverser une matrice ($X^T X$), ce qui devient <b>très coûteux</b> quand il y a beaucoup de features (des milliers, voire plus) — c'est $O(p^3)$ où $p$ est le nombre de features. C'est précisément pour cette raison qu'on utilise la <b>descente de gradient</b> en pratique dès que le problème grossit : c'est la vraie motivation de la suite de cette séance, pas juste une alternative académique.
</div>

### D'où vient cette formule ?

On sait qu'**au minimum, la pente (dérivée) vaut zéro** — pour $w$ et pour $b$ simultanément. Au lieu d'itérer, on peut donc directement poser :

$$\frac{\partial \text{MSE}}{\partial w} = 0 \qquad \text{et} \qquad \frac{\partial \text{MSE}}{\partial b} = 0$$

En écriture matricielle (avec $X$ incluant une colonne de 1 pour le biais), résoudre ce système donne directement :

$$X^TX\,\boldsymbol{\theta} = X^Ty \quad\Longrightarrow\quad \boldsymbol{\theta}^* = (X^TX)^{-1}X^Ty$$

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
Ce n'est pas une formule à admettre — c'est juste "poser gradient = 0" puis isoler $\theta$. Vérifions-la à la main sur un petit exemple avant de l'appliquer aux vraies données.
</div>

In [ ]:
# Exemple jouet à la main : 4 points
x_toy = np.array([1, 2, 3, 4])
y_toy = np.array([2.1, 3.9, 6.2, 7.8])

X_toy = np.c_[np.ones(len(x_toy)), x_toy]  # colonne de biais + x
print("X =\n", X_toy)

XT_X = X_toy.T @ X_toy
XT_y = X_toy.T @ y_toy
print("\nX^T X =\n", XT_X)
print("X^T y =", XT_y)

theta_toy = np.linalg.inv(XT_X) @ XT_y
print("\ntheta = (b, w) =", theta_toy.round(3))

In [ ]:
# Vérification : le résidu est orthogonal à l'espace des colonnes de X (X^T(y - Xθ*) = 0)
residu = y_toy - X_toy @ theta_toy
print("X^T . résidu =", X_toy.T @ residu)
print("Orthogonal (≈ 0) :", np.allclose(X_toy.T @ residu, 0))

<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
On retrouve $b \approx 0.15$ et $w \approx 1.94$ — la droite qui passe au mieux par ces 4 points, obtenue en un seul calcul matriciel. Passons maintenant aux vraies données California Housing.
</div>

In [ ]:
def equations_normales(X, y):
    n = X.shape[0]
    X_b = np.c_[np.ones((n, 1)), X]  # on ajoute la colonne de biais
    theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
    return theta  # [b, w]

theta_optimal = equations_normales(X_simple, y)
b_optimal, w_optimal = theta_optimal[0], theta_optimal[1]

print(f"w optimal = {w_optimal:.4f}")
print(f"b optimal = {b_optimal:.4f}")
print(f"MSE optimale = {mse(w_optimal, b_optimal, X_simple, y):.4f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_simple, y, alpha=0.15, s=15, label="données")
y_line_optimal = w_optimal * x_line + b_optimal
plt.plot(x_line, y_line_optimal, color="green", linewidth=2.5, label="droite optimale (équations normales)")
plt.xlabel("Revenu médian (10k $)")
plt.ylabel("Valeur médiane du logement (100k $)")
plt.legend()
plt.title("La vraie meilleure droite")
plt.show()

<div style="background:#f3e5f5;border-left:5px solid #8e24aa;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✏️ Exercice</b><br>
Compare cette MSE optimale à celle que tu avais trouvée à tâtons plus haut. De combien étais-tu loin ?
</div>

## 5. La descente de gradient

Les équations normales sont élégantes mais chères en calcul à grande échelle. La **descente de gradient** est une méthode *itérative* : on part d'un point au hasard sur la surface de perte, et on avance petit à petit dans la direction qui fait le plus baisser la perte.

### Le gradient

Le gradient de la MSE par rapport à $w$ et $b$ donne la direction de la **plus forte pente** :

$$\frac{\partial \text{MSE}}{\partial w} = -\frac{2}{n} \sum_{i=1}^{n} x_i \left( y_i - \hat{y}_i \right)$$

$$\frac{\partial \text{MSE}}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)$$

### La règle de mise à jour

À chaque itération, on avance dans la direction **opposée** au gradient (puisqu'on veut descendre, pas monter), avec un pas contrôlé par le **learning rate** $\alpha$ :

$$w \leftarrow w - \alpha \frac{\partial \text{MSE}}{\partial w} \qquad b \leftarrow b - \alpha \frac{\partial \text{MSE}}{\partial b}$$

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
Imagine que tu es dans le brouillard, sur la surface de perte, et que tu ne peux sentir que la pente sous tes pieds. Le gradient te dit "de quel côté ça monte" — tu fais donc un petit pas dans la direction opposée, tu re-sens la pente, tu refais un pas, etc. Le <b>learning rate</b> $\alpha$, c'est la taille de tes pas.
</div>

In [ ]:
def gradient_mse(w, b, X, y):
    n = len(y)
    y_pred = w * X.flatten() + b
    error = y - y_pred
    dw = -(2 / n) * np.sum(X.flatten() * error)
    db = -(2 / n) * np.sum(error)
    return dw, db

### Implémentation complète, pas à pas

In [ ]:
def descente_de_gradient(X, y, w_init, b_init, learning_rate, n_iterations):
    w, b = w_init, b_init
    historique_loss = []
    historique_params = [(w, b)]

    for i in range(n_iterations):
        dw, db = gradient_mse(w, b, X, y)
        w = w - learning_rate * dw
        b = b - learning_rate * db

        historique_loss.append(mse(w, b, X, y))
        historique_params.append((w, b))

    return w, b, historique_loss, historique_params

<div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Attention</b><br>
Avant de lancer la descente de gradient, il est essentiel de <b>standardiser</b> les features (moyenne 0, écart-type 1). Sans ça, avec des échelles très différentes, le gradient devient instable et un seul learning rate ne convient à aucune des deux dimensions correctement. On voit ça concrètement juste en dessous.
</div>

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_simple)

w_final, b_final, loss_history, params_history = descente_de_gradient(
    X_scaled, y, w_init=0.0, b_init=0.0, learning_rate=0.1, n_iterations=100
)

print(f"w trouvé par descente de gradient : {w_final:.4f}")
print(f"b trouvé par descente de gradient : {b_final:.4f}")
print(f"MSE finale : {mse(w_final, b_final, X_scaled, y):.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel("Itération")
plt.ylabel("MSE")
plt.title("Convergence de la descente de gradient")
plt.show()

<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
La courbe de perte doit <b>décroître et se stabiliser</b> — c'est la signature d'une descente de gradient qui converge correctement. Si elle oscille sans se stabiliser, ou explose, c'est presque toujours un <b>learning rate trop élevé</b> (section suivante).
</div>

## 6. Le rôle du learning rate

Le learning rate $\alpha$ est probablement l'**hyperparamètre** le plus important de la descente de gradient (rappel : un *hyperparamètre* est un réglage qu'on choisit *avant* l'entraînement, contrairement aux *paramètres* $w, b$ que l'entraînement ajuste lui-même).

Comparons plusieurs valeurs :

In [ ]:
learning_rates = [0.005, 0.1, 0.3, 1.1]


<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>

- <b>Trop petit</b> (0.005) : ça descend, mais beaucoup trop lentement — il faudrait des milliers d'itérations.
- <b>Bien choisi</b> (0.1 à 0.3 ici) : convergence rapide et stable.
- <b>Trop grand</b> (1.1) : les pas sont si larges qu'on "saute" par-dessus le minimum à chaque itération, voire on s'en éloigne — la perte peut osciller ou diverger.

</div>


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
w_range2 = np.linspace(-1, 2.5, 100)
b_range2 = np.linspace(-1.5, 1.5, 100)
W2, B2 = np.meshgrid(w_range2, b_range2)
Z2 = np.array([[mse(W2[i,j], B2[i,j], X_scaled, y) for j in range(W2.shape[1])] for i in range(W2.shape[0])])

for ax, lr, titre in zip(axes, [0.1, 1.1], ["learning rate raisonnable", "learning rate trop grand"]):
    _, _, _, params_hist = descente_de_gradient(X_scaled, y, -0.5, -1.0, lr, n_iterations=30)
    ws = [p[0] for p in params_hist]
    bs = [p[1] for p in params_hist]

    ax.contour(W2, B2, Z2, levels=25, cmap="viridis", alpha=0.6)
    ax.plot(ws, bs, "o-", color="red", markersize=3, linewidth=1)
    ax.plot(ws[0], bs[0], "s", color="black", markersize=8, label="départ")
    ax.plot(ws[-1], bs[-1], "*", color="gold", markersize=15, label="arrivée")
    ax.set_xlabel("w")
    ax.set_ylabel("b")
    ax.set_title(titre)
    ax.legend()

plt.tight_layout()
plt.show()


<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
La trajectoire de gauche descend proprement vers le centre du bol. Celle de droite "rebondit" d'un bord à l'autre de la vallée au lieu de converger — c'est visuellement ce que signifie "diverger". Choisir le learning rate est un compromis vitesse/stabilité, et il n'y a pas de valeur universelle : ça dépend du problème (d'où l'importance de standardiser les features au préalable, pour rester dans une plage de learning rates raisonnable).
</div>

<div style="background:#f3e5f5;border-left:5px solid #8e24aa;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✏️ Exercice</b><br>
Modifie la cellule des <code>learning_rates</code> ci-dessus : essaie une valeur encore plus grande (ex: 1.5) et observe ce qui se passe sur le graphe de convergence. Que se passe-t-il si le learning rate est négatif ?
</div>

## 7. Généraliser à plusieurs features

Jusqu'ici on a utilisé une seule feature (`MedInc`) pour rester visualisable en 2D/3D. En pratique, on utilise **toutes** les features disponibles. La formule devient :

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_p x_p + b = \mathbf{w}^T \mathbf{x} + b$$

Le principe (loss, gradient, mise à jour) reste **exactement le même** — seule la dimension change. C'est là que la formulation matricielle prend tout son sens : le code ne change quasiment pas.

<div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Attention</b><br>
On ne réimplémente pas la version multi-features à la main ici — l'objectif pédagogique (comprendre gradient descent) est déjà atteint avec le cas 1D. En pratique, <b>on utilise scikit-learn</b>, qui est optimisé, testé, et gère tous les cas limites.
</div>

In [ ]:
# Toutes les features cette fois
X_full = df.drop(columns=["MedHouseVal"])
y_full = df["MedHouseVal"].values

X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

scaler_full = StandardScaler()
X_train_scaled = scaler_full.fit_transform(X_train)
X_test_scaled = scaler_full.transform(X_test)

print("Train :", X_train_scaled.shape, " Test :", X_test_scaled.shape)

<div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Attention</b><br>
On standardise avec <code>fit_transform</code> sur le <b>train</b> uniquement, puis <code>transform</code> (sans re-fit) sur le <b>test</b>. Réutiliser les statistiques du train évite une fuite d'information (<i>data leakage</i>) — le modèle ne doit jamais "voir" d'information calculée sur le test, même indirectement. On y reviendra en détail en Séance 4.
</div>

In [ ]:
# Solution analytique via scikit-learn (LinearRegression = équations normales en interne)
modele_analytique = LinearRegression()
modele_analytique.fit(X_train_scaled, y_train)

print("MSE train :", np.mean((y_train - modele_analytique.predict(X_train_scaled)) ** 2))
print("MSE test  :", np.mean((y_test - modele_analytique.predict(X_test_scaled)) ** 2))

<div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Attention</b><br>
La descente de gradient codée à la main plus haut est une version <b>full-batch</b> : chaque pas utilise <b>tous</b> les exemples d'entraînement. <code>SGDRegressor</code> fait en réalité une descente <b>stochastique</b> (un exemple à la fois, par défaut) — un algorithme différent, dans la même famille. Les deux convergent en général vers une solution proche, mais ce n'est pas <i>la même</i> méthode.
</div>

In [ ]:
# Descente de gradient via scikit-learn (SGDRegressor)
modele_sgd = SGDRegressor(learning_rate="constant", eta0=0.01, max_iter=1000, random_state=42)
modele_sgd.fit(X_train_scaled, y_train)

print("MSE train (SGD) :", np.mean((y_train - modele_sgd.predict(X_train_scaled)) ** 2))
print("MSE test  (SGD) :", np.mean((y_test - modele_sgd.predict(X_test_scaled)) ** 2))

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
Les deux méthodes arrivent à des résultats très proches — c'est rassurant : ça confirme que ta compréhension "à la main" du cas 1D se généralise bien. Sur ce problème, <code>LinearRegression</code> (équations normales) est largement assez rapide vu le nombre de features ; <code>SGDRegressor</code> montre son intérêt surtout à plus grande échelle (millions de lignes, réseaux de neurones, etc.).
</div>

In [ ]:
coefficients = pd.DataFrame({
    "feature": X_full.columns,
    "poids (w)": modele_analytique.coef_
}).sort_values("poids (w)", key=abs, ascending=False)

coefficients

<div style="background:#f3e5f5;border-left:5px solid #8e24aa;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✏️ Exercice</b><br>
Regarde le tableau des coefficients ci-dessus. Quelle feature a le plus d'influence sur le prix prédit (en valeur absolue) ? Est-ce cohérent avec ce qu'on avait observé en Séance 1 ?
</div>

## 8. Débrief

Ce qu'on a construit aujourd'hui :

- Une **fonction de perte** (MSE) qui transforme "quelle droite est la meilleure ?" en un problème d'optimisation numérique
- Une **solution analytique directe** (équations normales) — précise mais coûteuse à grande échelle
- Une **solution itérative** (descente de gradient) — moins immédiate mais scalable, et c'est le principe qui va revenir dans quasiment tous les modèles qu'on verra ensuite (régression logistique en S3, et bien au-delà du module — réseaux de neurones, etc.)
- L'importance du **learning rate** comme compromis vitesse/stabilité

<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
La régression linéaire elle-même est une brique relativement simple. Ce qui compte vraiment aujourd'hui, c'est le <b>principe</b> : perte → gradient → mise à jour itérative. La Séance 3 réutilise <i>exactement</i> ce principe, avec une fonction de perte différente (log-loss) pour la classification.
</div>

## 9. Auto-évaluation rapide

1. Pourquoi élève-t-on les erreurs au carré dans la MSE plutôt que de les sommer directement ?
2. Que représente géométriquement le gradient de la fonction de perte ?
3. Que se passe-t-il si le learning rate est trop élevé ? Trop faible ?
4. Pourquoi standardise-t-on les features avant la descente de gradient ?